In [1]:
# !wget "https://archive.ics.uci.edu/ml/machine-learning-databases/00602/DryBeanDataset.zip"

In [2]:
import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns
import os
import pathlib

2026-02-23 18:24:56.484917: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771871096.662504      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771871096.716602      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771871097.156941      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771871097.157052      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771871097.157059      55 computation_placer.cc:177] computation placer alr

In [ ]:
class BeanDataPipeline:
    """Handles loading, cleaning, and transforming the Dry Bean Dataset."""
    def __init__(self, url, cache_dir):
        self.url = url
        self.cache_dir = cache_dir
        self.scaler = StandardScaler()
        self.label_encoder = LabelEncoder()

    def prepare_data(self):
        # Download and load
        # "extract=True" creates a directory for the data

        # path = tf.keras.utils.get_file("DryBeanDataset.zip", self.url, extract=True)
        
        path = tf.keras.utils.get_file(
            fname='DryBeanDataset.zip', 
            origin=self.url, 
            extract=True,
            cache_dir=self.cache_dir,
            cache_subdir='datasets'
        )

        data_dir = pathlib.Path(path) / 'DryBeanDataset'

        # This avoids the IsADirectoryError
        files = list(data_dir.glob('*'))
        print(f"Contents of directory: {files} @ {data_dir}")
        
        # Access the specific file (e.g., the Excel file in Dry Bean Dataset)
        bean_data_path = data_dir / 'Dry_Bean_Dataset.xlsx'
        
        if bean_data_path.exists():
            print(f"Success! Found file at: {bean_data_path}")
        # print(path)        
        df = pd.read_excel(bean_data_path)
        
        # Features and Labels
        X = df.drop('Class', axis=1).values
        y = df['Class'].values
        
        # Encoding and Scaling
        y_encoded = self.label_encoder.fit_transform(y)
        X_train, X_test, y_train, y_test = train_test_split(
            X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
        )
        
        self.X_train = self.scaler.fit_transform(X_train)
        self.X_test = self.scaler.transform(X_test)
        self.y_train = y_train
        self.y_test = y_test
        
        return self.X_train, self.X_test, self.y_train, self.y_test

In [7]:
home_dir = os.path.expanduser('~')
my_cache = os.path.join(home_dir, '.keras_cache')
my_cache

'/root/.keras_cache'

In [8]:
# 1. Pipeline and Data Prep

# cache_dir = '/kaggle/working/'
URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00602/DryBeanDataset.zip"

pipeline = BeanDataPipeline(URL, my_cache)
X_train, X_test, y_train, y_test = pipeline.prepare_data()

3620864/Unknown 0s 0us/stepContents of directory: [PosixPath('/tmp/.keras/datasets/DryBeanDataset_extracted/DryBeanDataset/Dry_Bean_Dataset.xlsx'), PosixPath('/tmp/.keras/datasets/DryBeanDataset_extracted/DryBeanDataset/Dry_Bean_Dataset.arff'), PosixPath('/tmp/.keras/datasets/DryBeanDataset_extracted/DryBeanDataset/Dry_Bean_Dataset.txt')] @ /tmp/.keras/datasets/DryBeanDataset_extracted/DryBeanDataset
Success! Found file at: /tmp/.keras/datasets/DryBeanDataset_extracted/DryBeanDataset/Dry_Bean_Dataset.xlsx


In [9]:
X_train.shape, y_train.shape

((10888, 16), (10888,))

In [10]:
class BeanClassifier(tf.keras.Model):
    """Custom TensorFlow Model for Multiclass Classification."""
    def __init__(self, num_classes):
        super(BeanClassifier, self).__init__()
        self.dense1 = tf.keras.layers.Dense(128, activation='relu')
        self.bn1 = tf.keras.layers.BatchNormalization()
        self.dropout1 = tf.keras.layers.Dropout(0.3)
        
        self.dense2 = tf.keras.layers.Dense(64, activation='relu')
        self.bn2 = tf.keras.layers.BatchNormalization()
        
        self.output_layer = tf.keras.layers.Dense(num_classes, activation='softmax')

    def call(self, inputs, training=False):
        x = self.dense1(inputs)
        x = self.bn1(x, training=training)
        x = tf.nn.relu(x)
        x = self.dropout1(x, training=training)
        
        x = self.dense2(x)
        x = self.bn2(x, training=training)
        x = tf.nn.relu(x)
        
        return self.output_layer(x)


In [11]:
# 2. Model Initialization
num_bean_types = len(pipeline.label_encoder.classes_)
model = BeanClassifier(num_classes=num_bean_types)

# 3. Compile
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

I0000 00:00:1771871151.655621      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


In [12]:
# 4. Training with Early Stopping
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

print("Starting OOP-based Training...")
history = model.fit(
    X_train, y_train, 
    epochs=50, 
    batch_size=512, 
    validation_split=0.15,
    callbacks=[early_stop],
    verbose=1
)


Starting OOP-based Training...
Epoch 1/50


I0000 00:00:1771871154.440709     125 service.cc:152] XLA service 0x7b2d8000baa0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1771871154.440743     125 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1771871154.799842     125 cuda_dnn.cc:529] Loaded cuDNN version 91002


 1/19 ━━━━━━━━━━━━━━━━━━━━ 1:21 5s/step - accuracy: 0.1387 - loss: 2.3187

I0000 00:00:1771871156.295944     125 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


19/19 ━━━━━━━━━━━━━━━━━━━━ 7s 126ms/step - accuracy: 0.3708 - loss: 1.7364 - val_accuracy: 0.7081 - val_loss: 1.2890
Epoch 2/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7845 - loss: 0.8292 - val_accuracy: 0.7772 - val_loss: 1.0784
Epoch 3/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8504 - loss: 0.6326 - val_accuracy: 0.8127 - val_loss: 0.9917
Epoch 4/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8846 - loss: 0.5019 - val_accuracy: 0.8403 - val_loss: 0.9206
Epoch 5/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8979 - loss: 0.4146 - val_accuracy: 0.8666 - val_loss: 0.8479
Epoch 6/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9044 - loss: 0.3545 - val_accuracy: 0.8917 - val_loss: 0.7593
Epoch 7/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9121 - loss: 0.3099 - val_accuracy: 0.9039 - val_loss: 0.6653
Epoch 8/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9174 - loss: 0.2744 - val_accuracy: 0.9106 - val_loss: 0.5886
E

In [13]:
# 5. Advanced Evaluation
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=pipeline.label_encoder.classes_))

86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step

--- Classification Report ---
              precision    recall  f1-score   support

    BARBUNYA       0.94      0.91      0.93       265
      BOMBAY       1.00      1.00      1.00       104
        CALI       0.95      0.94      0.95       326
    DERMASON       0.90      0.94      0.92       709
       HOROZ       0.96      0.96      0.96       386
       SEKER       0.95      0.96      0.96       406
        SIRA       0.89      0.85      0.87       527

    accuracy                           0.93      2723
   macro avg       0.94      0.94      0.94      2723
weighted avg       0.93      0.93      0.93      2723



In [14]:
print(f"Prediction Accuracy: {accuracy_score(y_pred, y_test):.2f}")

Prediction Accuracy: 0.93
